# 0. Sobre — Atlas dos Municípios

**Objetivo:** realizar um diagnóstico inicial de estrutura, qualidade, cobertura e distribuição dos dados, antes da integração das bases do Tech Challenge 3.

**Dicionário de dados:** preencher com as definições das variáveis relevantes após inspecionar a base.

**Perguntas norteadoras:** a estrutura está consistente? Há problemas de qualidade? A cobertura é adequada? Há valores atípicos que exigem investigação?

# 1. Configuração e importação de bibliotecas

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

# 2. Carga e inspeção dos dados

Verificar dimensões, tipos de colunas e primeiros registros.

In [8]:
caminho = Path("../data/raw/gold/indicadores_alunos")

df = pd.read_parquet(caminho)

print(f"Linhas: {len(df):,} | Colunas: {df.shape[1]}")
df.columns

Linhas: 12,431 | Colunas: 22


Index(['rede', 'id_municipio', 'total_alunos', 'total_presentes',
       'total_ausentes', 'total_alfabetizados', 'total_nao_alfabetizados',
       'total_cadernos_preenchidos', 'proficiencia_media',
       'proficiencia_minima', 'proficiencia_maxima', 'peso_aluno_medio',
       'nome', 'sigla_uf', 'nome_uf', 'nome_regiao', 'rede_descricao',
       'taxa_presenca', 'taxa_alfabetizados', 'taxa_preenchimento_caderno',
       'gold_processed_at', 'ano'],
      dtype='object')

In [3]:
caminho = Path("../data/raw/external/atlas/municipio.parquet")

df = pd.read_parquet(caminho)

print(f"Linhas: {len(df):,} | Colunas: {df.shape[1]}")
display(df.head())
df.info()

Linhas: 16,695 | Colunas: 230


,ano,id_municipio,expectativa_vida,fecundidade_total,mortalidade_1,mortalidade_5,razao_dependencia,prob_sobrevivencia_40,prob_sobrevivencia_60,taxa_envelhecimento,...,pia,pia_10_14,pia_15_17,pia_18_mais,indice_escolaridade,indice_frequencia_escolar,idhm,idhm_e,idhm_l,idhm_r
0,1991,1100015,62.01,4.08,45.58,58.05,73.50,83.81,66.87,1.82,...,NaN,NaN,NaN,NaN,0.117,0.109,0.329,0.112,0.617,0.516
1,1991,1100023,66.02,3.72,32.39,41.41,69.97,88.08,74.23,1.82,...,NaN,NaN,NaN,NaN,0.200,0.199,0.432,0.199,0.684,0.593
2,1991,1100031,63.16,3.89,41.52,52.94,77.23,85.09,69.00,2.59,...,NaN,NaN,NaN,NaN,0.111,0.106,0.309,0.108,0.636,0.430
3,1991,1100049,65.03,3.81,35.37,45.19,71.11,87.08,72.44,2.47,...,NaN,NaN,NaN,NaN,0.176,0.169,0.407,0.171,0.667,0.593
4,1991,1100056,62.73,3.55,43.00,54.82,71.09,84.62,68.20,2.46,...,NaN,NaN,NaN,NaN,0.152,0.175,0.386,0.167,0.629,0.547


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16695 entries, 0 to 16694
Columns: 230 entries, ano to idhm_r
dtypes: Int64(62), float64(167), object(1)
memory usage: 30.3+ MB


# 3. Qualidade dos dados

Consolidar valores ausentes e duplicatas em uma única verificação. Diferenciar duplicata exata de repetição de chave de negócio.

In [ ]:
qualidade = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "nulos_%": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique(dropna=True)
}).sort_values("nulos_%", ascending=False)

display(qualidade)
print("Linhas duplicadas exatas:", df.duplicated().sum())

## 3.1 Chaves e regras de negócio

Validar somente as chaves que representem efetivamente a granularidade da base. **A chave abaixo é uma hipótese de trabalho e deve ser confirmada com o dicionário de dados.**

In [ ]:
chave_proposta = []
if chave_proposta and all(c in df.columns for c in chave_proposta):
    print("Nulos na chave:", df[chave_proposta].isna().any(axis=1).sum())
    print("Repetições da chave proposta:", df.duplicated(chave_proposta).sum())
else:
    print("Chave não definida ou colunas ausentes: revisar granularidade antes do diagnóstico.")

In [ ]:
numericas = df.select_dtypes(include="number").columns
if len(numericas):
    display(df[numericas].describe().T.round(2))
else:
    print("Nenhuma coluna numérica identificada.")

### Observações de qualidade

- **Inconsistências identificadas:** preencher após executar.
- **Regras adicionais:** aplicar conforme definição oficial dos indicadores; não tratar automaticamente zeros ou negativos como erros.

# 4. Análise exploratória dos dados

Verificar cobertura e visualizar somente os indicadores úteis para esta base.

In [ ]:
dimensoes = ['ano', 'id_municipio']
for coluna in dimensoes:
    if coluna in df.columns:
        valores = df[coluna].dropna()
        print(f"{coluna}: {valores.nunique():,} valores distintos")
        if coluna == "ano":
            display(df[coluna].value_counts(dropna=False).sort_index().rename("registros"))
        else:
            display(df[coluna].value_counts(dropna=False).head(10).rename("registros"))

## 4.1 Distribuição de indicadores socioeconômicos

Selecionar um indicador representativo, evitando múltiplos gráficos com a mesma informação.

In [ ]:
indicador = 'expectativa_vida'  # substituir se a coluna tiver outro nome
if indicador and indicador in df.columns and pd.api.types.is_numeric_dtype(df[indicador]):
    display(df[indicador].describe().round(2))
    plt.figure(figsize=(8, 4))
    sns.histplot(data=df, x=indicador, bins=25)
    plt.title(f"Distribuição — {indicador}")
    plt.tight_layout()
    plt.show()
else:
    print("Defina um indicador numérico existente para visualizar sua distribuição.")

## 4.2 Validações específicas

Investigar, quando aplicável, percentuais fora da escala, códigos inválidos, cobertura por ano, denominadores ausentes, valores incompatíveis e outliers relevantes. **Evitar regras genéricas não justificadas.**

In [ ]:
# Exemplo de regra: habilite APENAS se o indicador for uma taxa na escala 0–100.
# coluna_taxa = "taxa_alfabetizacao"
# if coluna_taxa in df.columns:
#     inconsistentes = df[coluna_taxa].notna() & ~df[coluna_taxa].between(0, 100)
#     print("Taxas fora do intervalo [0, 100]:", inconsistentes.sum())

# 5. Conclusão geral

**Estrutura e cobertura:** preencher com os achados reais.

**Qualidade dos dados:** listar somente problemas efetivamente encontrados.

**Impacto para a integração:** registrar chaves, diferenças de anos, nível de agregação e ajustes necessários.

**Decisão:** pronta para integração / exige investigação / exige tratamento (definir após execução).